# 05 — Validação das matrizes de conectividade
Checagens: existência, formato, simetria, diagonal, valores finitos, intervalo [-1, 1] e número de volumes (timepoints) por série.

In [ ]:
import numpy as np
import pandas as pd
import config

df = pd.read_csv(config.ARQ_PARTICIPANTES, dtype={"ID_padronizado": str, "subject_id": str})
print("Participantes:", len(df))

In [ ]:
linhas = []
for _, r in df.iterrows():
    f = config.CONECT / f"FC_{r['ID_padronizado']}.npy"
    reg = {"ID": r["ID_padronizado"], "site": r["site"], "existe": f.exists()}
    if f.exists():
        fc = np.load(f)
        reg.update(
            shape_ok=fc.shape == (config.N_ROIS, config.N_ROIS),
            simetrica=bool(np.allclose(fc, fc.T)),
            diag_ok=bool(np.allclose(np.diag(fc), 1.0)),
            finita=bool(np.isfinite(fc).all()),
            faixa_ok=bool(fc.min() >= -1.0001 and fc.max() <= 1.0001),
        )
    with open(r["caminho"]) as fh:
        reg["n_volumes"] = sum(1 for _ in fh) - 1   # menos o cabeçalho
    linhas.append(reg)

val = pd.DataFrame(linhas)
checks = ["existe", "shape_ok", "simetrica", "diag_ok", "finita", "faixa_ok"]
print(val[checks].fillna(False).all())
print("\nParticipantes com alguma falha:", int((~val[checks].fillna(False).all(axis=1)).sum()))

In [ ]:
print("Volumes por série (resumo):")
print(val["n_volumes"].describe())
display(val.groupby("site")["n_volumes"].agg(["count", "min", "median", "max"]))

Séries muito curtas geram correlações ruidosas. Observe sites com poucos volumes antes de interpretar diferenças de conectividade.

In [ ]:
val.to_csv(config.PROCESSADO / "validacao.csv", index=False)
print("Salvo em", config.PROCESSADO / "validacao.csv")